# Real Estate Analysis
## Machine Learning & Prediction
* **Goal:** Develop a machine learning model for price prediction based on property characteristics and location

In [17]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error, 
    mean_squared_error, 
    r2_score
)
from sklearn.dummy import DummyClassifier

In [18]:
# Safe Dataloading
real_estate_ml = '../data/real_estate_ml.csv'

try:
    df = pd.read_csv(real_estate_ml)
    print(f'File Loaded successfully')
except FileNotFoundError:
    print(f'File Not Found - check {real_estate_ml}')
except Exception as e:
    print(f'Error occured - check {e}')

File Loaded successfully


In [19]:
# Check the head
df.head()

,country,location,building_construction_year_analysis,building_total_floors,apartment_floor_analysis,apartment_rooms_analysis,apartment_bedrooms_analysis,apartment_bathrooms_analysis,apartment_total_area_analysis,apartment_rooms_missing,apartment_bedrooms_missing,apartment_bathrooms_missing,building_construction_year_missing,apartment_floor_missing,building_total_floors_missing,price_in_USD
0,Turkey,"Mediterranean Region, Turkey",2022.0,5.0,1.0,3.0,2.0,2.0,120.0,0,0,0,1,0,0,315209.0
1,Turkey,"Kalkan, Mediterranean Region, Kas, Turkey",2021.0,2.0,4.0,2.0,2.0,1.0,500.0,1,1,1,0,1,0,1108667.0
2,Turkey,"Mediterranean Region, Antalya, Turkey",2022.0,5.0,2.0,2.0,1.0,1.0,65.0,0,0,0,1,0,0,173211.0
3,Thailand,"Chon Buri Province, Pattaya, Thailand",2020.0,15.0,5.0,2.0,1.0,1.0,86.0,0,0,0,0,0,0,99900.0
4,Thailand,"Chon Buri Province, Pattaya, Thailand",2026.0,8.0,3.0,3.0,2.0,1.0,86.0,0,0,0,0,0,0,67000.0


In [20]:
# Define features and target
X = df.drop('price_in_USD', axis=1)
y = df['price_in_USD']

In [21]:
# Split data into training and test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [22]:
# Check the shape of each set
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(115968, 15)
(28993, 15)
(115968,)
(28993,)


In [23]:
# Check location before encoding
print(X_train['location'].nunique())
print(X_train['location'].value_counts().head(10))

6546
location
Mediterranean Region, Sekerhane Mahallesi, Alanya, Turkey                                                5797
Central Federal District, poselenie Sosenskoe, Novomoskovsky Administrative Okrug, Russia                5583
Central Hungary, Budapest, Komarom-Esztergom, Hungary                                                    3514
Minsk, Belarus                                                                                           3341
Kommunarka, Central Federal District, poselenie Sosenskoe, Novomoskovsky Administrative Okrug, Russia    2171
Transdanubia, Budapest, Komarom-Esztergom, Hungary                                                       2030
Dubai, UAE                                                                                               1963
Montenegro                                                                                               1733
Brest Region, Brest, Belarus                                                                             1

### Encoding categorical features

Machine learning models cannot directly work with text-based categorical features.

For `country`, One-Hot Encoding is suitable because there are only 27 different categories.

For `location`, there are more than 6,500 categories. One-Hot Encoding would create thousands of additional columns. Therefore, frequency encoding is used to represent each location by how frequently it occurs in the training data.

In [24]:
# Frequency encoding for location

location_frequency = X_train['location'].value_counts()

X_train['location_frequency'] =  X_train['location'].map(location_frequency)
X_test['location_frequency'] =  X_test['location'].map(location_frequency)

X_train['location_frequency'] =  X_train['location_frequency'].fillna(0)
X_test['location_frequency'] =  X_test['location_frequency'].fillna(0)

In [25]:
# One Hot Encoding for country

X_train = pd.get_dummies(
    X_train, 
    columns=['country'], 
    dtype=int
)

X_test = pd.get_dummies(
    X_test, 
    columns=['country'], 
    dtype=int
)

In [26]:
# Align X_test and X_train country column

X_train, X_test = X_train.align(
    X_test, 
    join='left', 
    axis=1, 
    fill_value=0
)

In [27]:
# Check shape 

print(X_train.shape)
print(X_test.shape)
print(X_train.dtypes.value_counts())

(115968, 43)
(28993, 43)
int64      35
float64     7
object      1
Name: count, dtype: int64


In [28]:
# Remove original location column

X_train = X_train.drop('location', axis=1)
X_test = X_test.drop('location', axis=1)

In [29]:
print(X_train.shape)
print(X_test.shape)
print(X_train.dtypes.value_counts())

(115968, 42)
(28993, 42)
int64      35
float64     7
Name: count, dtype: int64


In [30]:
# Start with Baseline Model
from sklearn.dummy import DummyRegressor

In [31]:
# Create Baseline Model

baseline = DummyRegressor(strategy='mean')

baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)

In [32]:
# Evaluate the basline

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

print(f'Baseline MAE: $ {baseline_mae:.2f}')
print(f'Baseline RMSE: $ {baseline_rmse:.2f}')
print(f'Baseline R^2: $ {baseline_r2:.4f}')

Baseline MAE: $ 373029.78
Baseline RMSE: $ 808264.00
Baseline R^2: $ -0.0000


## Baseline Model

A baseline model was created using the mean property price.

The baseline provides a reference point for evaluating more complex regression models. It predicts the same average price for every property and therefore does not use the available property features.

The baseline achieved an MAE of approximately **$373,030**, an RMSE of approximately **$808,264**, and an R² of approximately **0**.

More advanced models should improve on these results by using the information contained in the property features.

In [33]:
# Create new Model - Linear Regression
from sklearn.linear_model import LinearRegression

In [34]:
linear = LinearRegression()

linear.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [35]:
linear_pred = linear.predict(X_test)

In [36]:
linear_pred[:5]

array([ 804473.44263946, 1206655.30645833,  300515.85516478,
       1252999.70034584,   96298.7162857 ])

In [37]:
# Evaluate the model
linear_mae = mean_absolute_error(y_test, linear_pred)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_pred))
linear_r2 = r2_score(y_test, linear_pred)

print(f'Linear Regression MAE: $ {linear_mae:.2f}')
print(f'Linear Regression RMSE: $ {linear_rmse:.2f}')
print(f'Linear Regression R^2: $ {linear_r2:.4f}')

Linear Regression MAE: $ 295397.93
Linear Regression RMSE: $ 706736.93
Linear Regression R^2: $ 0.2354


In [38]:
# Compare Linear Model with Baseline Model
comparison = pd.DataFrame({
    'Model': ['Baseline(Mean)', 'Linear Regression'], 
    'MAE': [baseline_mae, linear_mae], 
    'RMSE': [baseline_rmse, linear_rmse], 
    'R^2': [baseline_r2, linear_r2]
})
comparison

,Model,MAE,RMSE,R^2
0,Baseline(Mean),373029.776575,808264.004638,-0.000025
1,Linear Regression,295397.934180,706736.926762,0.235425


## Linear Regression Results

The Linear Regression model performs better than the baseline model.

Compared to the baseline, the average prediction error (MAE) decreased from approximately **$373,030** to **$295,398**. The RMSE also decreased from approximately **$808,264** to **$706,737**.

The model achieved an R² score of **0.2354**. This means that the available property features explain approximately **23.5%** of the variation in property prices.

Although the Linear Regression model improves the predictions, real estate prices likely contain non-linear relationships. More advanced models may therefore achieve better results.

In [39]:
# Implement RandomForestRegressor
from sklearn.ensemble import RandomForestRegressor

In [40]:
random_forest = RandomForestRegressor(
    n_estimators=100, 
    random_state=42, 
    n_jobs=1
)

random_forest.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [41]:
random_forest_pred = random_forest.predict(X_test)

In [26]:
random_forest_pred[:5]

array([ 499303.39      , 2138891.685     ,  226545.75966667,
       1434986.925     ,  190452.56      ])

In [27]:
# Evaluate the model
rf_mae = mean_absolute_error(y_test, random_forest_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, random_forest_pred))
rf_r2 = r2_score(y_test, random_forest_pred)

print(f'Random Forest MAE: $ {rf_mae:.2f}')
print(f'Random Forest RMSE: $ {rf_rmse:.2f}')
print(f'Random Forest R^2: $ {rf_r2:.4f}')

Random Forest MAE: $ 180263.93
Random Forest RMSE: $ 560308.82
Random Forest R^2: $ 0.5194


## Random Forest Results

The Random Forest model achieved the best performance so far.

It reduced the MAE to approximately **$180,264** and the RMSE to approximately **$560,309**. The model achieved an R² score of **0.5194**, meaning that it explains approximately **51.9%** of the variation in property prices.

The Random Forest performs substantially better than both the baseline model and the Linear Regression model. This suggests that non-linear relationships between property characteristics and price are important.

In [29]:
# Import Cross Validation
from sklearn.model_selection import cross_validate

In [ ]:
# Cross Validation for RF Model
rf_cv_results = cross_validate(
    random_forest, 
    X_train, 
    y_train, 
    cv=5,
    scoring={
        'mae': 'neg_mean_absolute_error', 
        'rmse': 'neg_root_mean_squared_error', 
        'r2': 'r2'
    }, 
    return_train_score=False
)

In [32]:
# Create CV summary
cv_summary = pd.DataFrame({
    'Fold': range(1, 6), 
    'MAE': -rf_cv_results['test_mae'], 
    'RMSE': -rf_cv_results['test_rmse'], 
    'R^2': -rf_cv_results['test_r2']
})

cv_summary

,Fold,MAE,RMSE,R^2
0,1,193067.118755,663673.838258,-0.477003
1,2,185565.986416,604163.947991,-0.505076
2,3,187497.957697,572904.227937,-0.512384
3,4,181880.745923,580796.466005,-0.520705
4,5,186586.938917,570426.328393,-0.506085


In [33]:
print(f'Average CV MAE: $ {-rf_cv_results['test_mae'].mean():,.2f}')
print(f'Average CV RMSE: $ {-rf_cv_results['test_rmse'].mean():,.2f}')
print(f'Average CV R^2: $ {-rf_cv_results['test_r2'].mean():,.4f}')

Average CV MAE: $ 186,919.75
Average CV RMSE: $ 598,392.96
Average CV R^2: $ -0.5043


## Cross-Validation Adjustment

The initial cross-validation results showed an average R² score below zero. This indicates that the model performance was not stable across the five validation folds.

To create more representative folds, `KFold` cross-validation with shuffled data is used. Shuffling helps ensure that each fold contains a more similar mix of countries, locations, and property prices.

The `random_state=42` parameter ensures that the data is shuffled in the same way every time the notebook is executed. The test set remains untouched and will only be used for the final model evaluation.

In [34]:
from sklearn.model_selection import KFold

In [35]:
cv = KFold(
    n_splits=5,
    shuffle=True, 
    random_state=42
)

In [37]:
rf_cv_shuffled = cross_validate(
    random_forest, 
    X_train, 
    y_train, 
    cv=cv, 
    scoring={
        'mae': 'neg_mean_absolute_error', 
        'rmse': 'neg_root_mean_squared_error', 
        'r2': 'r2'
    },
    return_train_score=False
)

In [38]:
shuffled_cv_summary = pd.DataFrame({
    'Fold': range(1,6), 
    'MAE': -rf_cv_shuffled['test_mae'], 
    'RMSE': -rf_cv_shuffled['test_rmse'],
    'R^2': rf_cv_shuffled['test_r2']
})

shuffled_cv_summary

,Fold,MAE,RMSE,R^2
0,1,193448.632100,664812.599230,0.451065
1,2,184053.566073,565093.605366,0.544271
2,3,188665.598231,607674.523804,0.488093
3,4,185288.808309,590882.857922,0.509966
4,5,184387.387457,598240.905563,0.470278


In [40]:
# Evaluate the new mean
print(f'Shuffled CV MAE: $ {-rf_cv_shuffled['test_mae'].mean():.2f}')
print(f'Shuffled CV RMSE: $ {-rf_cv_shuffled['test_rmse'].mean():.2f}')
print(f'Shuffled CV R^2: $ {rf_cv_shuffled['test_r2'].mean():.4f}')

Shuffled CV MAE: $ 187168.80
Shuffled CV RMSE: $ 605340.90
Shuffled CV R^2: $ 0.4927


In [41]:
# Import GridSearch
from sklearn.model_selection import GridSearchCV

In [43]:
# Define the GridSearch
param_grid = {
    'n_estimators': [100, 200], 
    'max_depth': [None, 20], 
    'min_samples_leaf': [1, 2]
}

In [44]:
rf_for_tuning = RandomForestRegressor(
    random_state=42, 
    n_jobs=1
)

grid_search = GridSearchCV(
    estimator=rf_for_tuning, 
    param_grid=param_grid, 
    cv=cv, 
    scoring='r2', 
    n_jobs=1, 
    verbose=1
)

In [45]:
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 8 candidates, totalling 40 fits


,estimator,RandomForestR...ndom_state=42)
,param_grid,"{'max_depth': [None, 20], 'min_samples_leaf': [1, 2], 'n_estimators': [100, 200]}"
,scoring,'r2'
,n_jobs,1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,200


In [46]:
# Check the best parameters
print('Best Parameters:')
print(grid_search.best_params_)
print(f'\nBest cross-validation R^2: {grid_search.best_score_:.4f}')

Best Parameters:
{'max_depth': None, 'min_samples_leaf': 2, 'n_estimators': 200}

Best cross-validation R^2: 0.5188


In [48]:
# Make prediction with best parameters
best_rf = grid_search.best_estimator_
best_rf_pred = best_rf.predict(X_test)

In [50]:
# Evaluate the tuned model

best_rf_mae = mean_absolute_error(y_test, best_rf_pred)
best_rf_rmse = np.sqrt(mean_squared_error(y_test, best_rf_pred))
best_rf_r2 = r2_score(y_test, best_rf_pred)

print(f'Optimized Random Forest MAE: $ {best_rf_mae:.2f}')
print(f'Optimized Random Forest RMSE: $ {best_rf_rmse:.2f}')
print(f'Optimized Random Forest R^2: $ {best_rf_r2:.4f}')

Optimized Random Forest MAE: $ 176917.72
Optimized Random Forest RMSE: $ 542322.42
Optimized Random Forest R^2: $ 0.5498


## Hyperparameter Tuning Results

The Random Forest model was optimized using `GridSearchCV` with 5-fold shuffled cross-validation.

The best hyperparameter combination was:

- `n_estimators=200`
- `max_depth=None`
- `min_samples_leaf=2`

The optimized model achieved an average cross-validation R² score of **0.5188**.

When evaluated on the untouched test set, the optimized Random Forest achieved:

- **MAE:** approximately **$176,918**
- **RMSE:** approximately **$542,322**
- **R²:** **0.5498**

The optimized model performs better than the original Random Forest baseline. Hyperparameter tuning reduced the prediction errors and increased the explained variation in property prices.

In [51]:
feature_importance = pd.DataFrame({
    'Features': X_train.columns, 
    'Importance': best_rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance', 
    ascending=False
)

feature_importance.head(15)

,Features,Importance
6,apartment_total_area_analysis,0.365820
13,location_frequency,0.166841
26,country_Italy,0.059044
5,apartment_bathrooms_analysis,0.055841
38,country_UAE,0.049968
1,building_total_floors,0.041080
35,country_Spain,0.034777
4,apartment_bedrooms_analysis,0.031101
0,building_construction_year_analysis,0.030728
3,apartment_rooms_analysis,0.027551


In [52]:
# Define second GridSearch
param_grid_2 = {
    'n_estimators': [200, 300], 
    'max_depth': [None], 
    'min_samples_leaf': [2, 3, 5],
    'max_features': [0.7, 1.0]
}

In [53]:
grid_search_2 = GridSearchCV(
    estimator=rf_for_tuning, 
    param_grid=param_grid_2, 
    cv=cv, 
    scoring='r2', 
    n_jobs=1, 
    verbose=1
)

grid_search_2.fit(X_train, y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


,estimator,RandomForestR...ndom_state=42)
,param_grid,"{'max_depth': [None], 'max_features': [0.7, 1.0], 'min_samples_leaf': [2, 3, ...], 'n_estimators': [200, 300]}"
,scoring,'r2'
,n_jobs,1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,200


In [54]:
# Evaluate best params
print('Best Parameters - Round 2')
print(grid_search_2.best_params_)
print(f'\nBest cross-validation R^2: round 2: {grid_search_2.best_score_:.4f}')

Best Parameters - Round 2
{'max_depth': None, 'max_features': 0.7, 'min_samples_leaf': 5, 'n_estimators': 200}

Best cross-validation R^2: round 2: 0.5369


In [55]:
# Define third GridSearch
param_grid_3 = {
    'n_estimators': [200], 
    'max_depth': [None], 
    'min_samples_leaf': [5, 8, 10],
    'max_features': [0.2, 0.5, 0.7, 0.9]
}

In [59]:
grid_search_3 = GridSearchCV(
    estimator=rf_for_tuning, 
    param_grid=param_grid_3, 
    cv=cv, 
    scoring='r2', 
    n_jobs=1, 
    verbose=1
)

grid_search_3.fit(X_train, y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


,estimator,RandomForestR...ndom_state=42)
,param_grid,"{'max_depth': [None], 'max_features': [0.2, 0.5, ...], 'min_samples_leaf': [5, 8, ...], 'n_estimators': [200]}"
,scoring,'r2'
,n_jobs,1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,200


In [61]:
# Evaluate best params
print('Best Parameters - Round 3')
print(grid_search_3.best_params_)
print(f'\nBest cross-validation R^2: round 2: {grid_search_3.best_score_:.4f}')

Best Parameters - Round 3
{'max_depth': None, 'max_features': 0.7, 'min_samples_leaf': 5, 'n_estimators': 200}

Best cross-validation R^2: round 2: 0.5369


## Final Random Forest Tuning

A third GridSearchCV round was conducted to further refine the Random Forest model.

Additional values for `max_features` and `min_samples_leaf` were tested. However, the best hyperparameter combination remained unchanged:

- `n_estimators=200`
- `max_depth=None`
- `min_samples_leaf=5`
- `max_features=0.7`

The best average cross-validation R² score remained **0.5369**, identical to the result from the second tuning round.

Since the third search did not improve model performance, further Random Forest tuning was stopped. The selected configuration will be used as the final optimized Random Forest model.

In [62]:
from xgboost import XGBRegressor

In [67]:
# Implement Baseline XGB Model

xgb_baseline = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=200,
    learning_rate=0.05, 
    max_depth=6, 
    random_state=42, 
    n_jobs=1, 
    tree_method='hist'
)

In [68]:
xgb_baseline.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [69]:
# Cross Validation on Baseline XGB
xgb_cv_results = cross_validate(
    xgb_baseline, 
    X_train, 
    y_train, 
    cv=cv, 
    scoring={
        'mae': 'neg_mean_absolute_error', 
        'rmse': 'neg_root_mean_squared_error', 
        'r2': 'r2'
    }, 
    return_train_score=False, 
    n_jobs=1
)

In [70]:
xgb_summary = pd.DataFrame({
    'Fold': range(1, 6), 
    'MAE': -xgb_cv_results['test_mae'], 
    'RMSE': -xgb_cv_results['test_rmse'], 
    'R^2': xgb_cv_results['test_r2']
})
xgb_summary

,Fold,MAE,RMSE,R^2
0,1,207257.302860,654323.374413,0.468250
1,2,199428.700249,573512.481900,0.530591
2,3,203980.404662,599363.420261,0.502000
3,4,201335.248639,597786.879384,0.498448
4,5,201798.622017,598621.294632,0.469604


In [71]:
print(f'XGBoost CV MAE: $ {-xgb_cv_results['test_mae'].mean():.2f}')
print(f'XGBoost CV RMSE: $ {-xgb_cv_results['test_rmse'].mean():.2f}')
print(f'XGBoost CV R^2: $ {xgb_cv_results['test_r2'].mean():.4f}')

XGBoost CV MAE: $ 202760.06
XGBoost CV RMSE: $ 604721.49
XGBoost CV R^2: $ 0.4938


In [74]:
# Hyperparemeter adjustment
xgb_param_grid_1 = {
    'n_estimators': [300, 500], 
    'learning_rate': [0.03, 0.06], 
    'max_depth': [4, 6, 8]
}

In [79]:
xgb_for_tuning = XGBRegressor(
    objective='reg:squarederror', 
    random_state=42, 
    n_jobs=1, 
    tree_method='hist'
)

xgb_grid_search_1 = GridSearchCV(
    estimator=xgb_for_tuning, 
    param_grid=xgb_param_grid_1, 
    cv=cv, 
    scoring='r2', 
    n_jobs=1, 
    verbose=1
)

In [80]:
xgb_grid_search_1.fit(X_train, y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


,estimator,"XGBRegressor(...ree=None, ...)"
,param_grid,"{'learning_rate': [0.03, 0.06], 'max_depth': [4, 6, ...], 'n_estimators': [300, 500]}"
,scoring,'r2'
,n_jobs,1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,objective,'reg:squarederror'


In [81]:
# Check best params
print('Best XGBoost parameters:')
print(xgb_grid_search_1.best_params_)
print(f'\nBest XGBoost CV R^2: {xgb_grid_search_1.best_score_:.4f}')

Best XGBoost parameters:
{'learning_rate': 0.06, 'max_depth': 6, 'n_estimators': 500}

Best XGBoost CV R^2: 0.5132


## XGBoost Results

An XGBoost Regressor was trained and evaluated using the same 5-fold shuffled cross-validation strategy as the Random Forest model.

The baseline XGBoost model achieved an average cross-validation R² score of **0.4938**.

After hyperparameter tuning with GridSearchCV, the best configuration was:

- `n_estimators=500`
- `learning_rate=0.06`
- `max_depth=6`

The tuned XGBoost model achieved an average cross-validation R² score of **0.5132**.

Although hyperparameter tuning improved XGBoost performance, the optimized Random Forest model remained the best-performing model with an average cross-validation R² score of **0.5369**.

In [14]:
# Transform target
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

In [15]:
print('Original price range:')
print(y_train.describe())
print('\nLog-transformed price range:')
print(y_train_log.describe())

Original price range:
count    1.159680e+05
mean     4.129803e+05
std      8.503475e+05
min      0.000000e+00
25%      1.055665e+05
50%      1.906210e+05
75%      3.989300e+05
max      3.060283e+07
Name: price_in_USD, dtype: float64

Log-transformed price range:
count    115968.000000
mean         12.244809
std           1.107131
min           0.000000
25%          11.567106
50%          12.158048
75%          12.896544
max          17.236603
Name: price_in_USD, dtype: float64


In [42]:
rf_log_model = RandomForestRegressor(
    n_estimators=200, 
    max_depth=None, 
    min_samples_leaf=5, 
    max_features=0.7, 
    random_state=42, 
    n_jobs=1
)
rf_log_model.fit(X_train, y_train_log)

,n_estimators,200
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,0.7
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False
